# Data Mining - Project

### Work done by
- Roberto Rolão de Castro 107133
- Tiago Caridade Gomes 108307

## Tracking the Evolution of E-commerce Websites in Portugal

### Description
Study how e-commerce websites in Portugal, such as those related to retail (e.g., Worten, Fnac), have changed in terms of offerings and product availability.

### Data
Gather versions of popular e-commerce sites over time from Arquivo.pt.

---

## Project Objectives

The main objectives of this project are:
- To understand the evolution of e-commerce websites in Portugal.
- To analyze changes in product availability, pricing, and diversity over time.

## State of the Art / Similar Works

Research on tracking changes in online content using web archives has been supported by the Arquivo.pt awards, which recognize innovative use cases involving archived web data. Some of these award-winning projects have tackled challenges similar to ours:

- **2024 Arquivo.pt Award Winners**: Projects that focused on extracting and analyzing data from Portuguese web archives to gain insights into historical trends in different domains. More details can be found [here](https://sobre.arquivo.pt/en/meet-the-winners-of-the-arquivo-pt-award-2024/).
  
- **2021 Arquivo.pt Award Winners**: Notable works from this year included analyses that utilized web archives for various purposes, including monitoring the evolution of online information. More details [here](https://sobre.arquivo.pt/en/meet-the-winners-of-the-arquivo-pt-award-2021/).

- **2020 Arquivo.pt Award Winners**: Projects from 2020 included using the Arquivo.pt for research on Portuguese historical data, similar to our interest in studying the evolution of e-commerce websites. More information [here](https://sobre.arquivo.pt/en/meet-the-winners-of-the-arquivo-pt-award-2020/).

These works demonstrate the versatility of using web archives like Arquivo.pt for historical analysis of online content, providing a foundation and inspiration for our project.

## Data Acquisition

The data was gathered using the [Arquivo.pt](https://arquivo.pt/) API to fetch archived versions of websites related to retail and technology products. We defined a set of keywords and a list of URLs for popular e-commerce sites.

**Methodology:**
- Defined search terms related to specific product categories (e.g., electronics, appliances).
- Selected major e-commerce platforms in Portugal (e.g., Worten, Fnac, Radio Popular).
- Implemented a script to iteratively request data from the Arquivo.pt API while handling pagination.

**Challenges:**
- **Pagination Limitations**: The API has limitations on how much data can be fetched at once. This was overcome by implementing a custom pagination strategy to ensure all data was retrieved.
- **Duplicate URLs**: To avoid redundant data, a mechanism was implemented to track already processed URLs.


## Implementation

Below is an overview of the Python code used to gather the data from Arquivo.pt. The code is divided into functions for reading input URLs, fetching data from the API, and processing the results. All code is commented to ensure clarity.

#### Query definition and methodology for organizing/saving data

---

In [ ]:
import requests
import pandas as pd
import os
from bs4 import BeautifulSoup
import re

def read_list_from_file(filename):
    """
    Reads a list of items from a file, one item per line.
    :param filename: The name of the file to read from.
    :return: A list of strings, each representing an item from the file.
    """
    with open(filename, 'r') as file:
        return [line.strip() for line in file if line.strip()]  # Read non-empty lines and remove whitespace

def fetch_all_results(websites, keywords):
    """
    Fetch all results from Arquivo.pt for the given websites and keywords.
    :param websites: List of websites to search.
    :param keywords: List of keywords to use in the search.
    """
    # Base URL for Arquivo.pt text search API
    url = "https://arquivo.pt/textsearch"
    all_data = []  # List to store data fetched from the API
    seen_urls = set()  # Set to track URLs that have already been added to avoid duplicates

    # Ensure the "Results" directory exists to save output files
    os.makedirs('./Results', exist_ok=True)

    # Iterate over each website and keyword combination
    for website in websites:
        for keyword in keywords:
            print(f"Starting search for '{keyword}' on {website}")
            # Define the parameters for the API request
            base_params = {
                "from": "20200101000000",  # Start date for the search (YYYYMMDDHHMMSS format)
                "to": "20240101000000",    # End date for the search (YYYYMMDDHHMMSS format)
                "maxItems": 500,            # Maximum number of items per request
                "q": keyword,               # Keyword to search for
                "siteSearch": website,      # Website to restrict the search to
                "dedupValue": 0,            # Disable deduplication by value
                "dedupField": "url",       # Field used for deduplication
                "offset": 0                 # Offset for pagination
            }

            # Fetch data for each combination of website and keyword
            while True:
                try:
                    # Make a GET request to the API with the defined parameters
                    response = requests.get(url, params=base_params)
                    response.raise_for_status()  # Raise an error for bad status codes
                except requests.exceptions.RequestException as e:
                    # Handle any request exceptions (e.g., connection issues, timeouts)
                    print(f"Error fetching data at offset {base_params['offset']}: {e}")
                    break

                # Parse the response JSON
                data = response.json()
                if 'response_items' in data and data['response_items']:
                    # Convert the response items to a DataFrame
                    current_data = pd.DataFrame(data['response_items'])

                    # Filter out duplicate URLs based on 'linkToExtractedText'
                    new_data = current_data[~current_data['linkToExtractedText'].isin(seen_urls)]
                    if not new_data.empty:
                        all_data.append(new_data)  # Append new data to the list
                        # Add the new URLs to the seen set to avoid duplicates
                        seen_urls.update(new_data['linkToExtractedText'])
                        print(f"Added {len(new_data)} new items for '{keyword}' on {website} at offset {base_params['offset']}")
                    else:
                        print(f"No new items found for '{keyword}' on {website} at offset {base_params['offset']}")

                    # If fewer results were returned than maxItems, it means we reached the last page
                    if len(current_data) < base_params['maxItems']:
                        print("Reached the last page of data.")
                        break

                    # Increment the offset to fetch the next batch of results
                    base_params['offset'] += base_params['maxItems']
                else:
                    # No more data available or reached API limit
                    print("No more data available or reached API limit.")
                    break

            # Save the results for the current website and keyword
            if all_data:
                # Concatenate all data frames into one final DataFrame
                final_data = pd.concat(all_data, ignore_index=True)
                
                # Drop duplicates from the final DataFrame based on 'linkToExtractedText'
                final_data = final_data.drop_duplicates(subset=['linkToExtractedText'])
                final_data = final_data.drop_duplicates(subset=['originalUrl'])

                # Define the output filename based on the website and keyword
                output_filename = f"./Results/{website.replace('https://', '').replace('.', '_')}_{keyword}.csv"
                # Save the DataFrame to a CSV file
                final_data.to_csv(output_filename, index=False)
                print(f"Data saved to {output_filename}")
                all_data.clear()  # Clear the list for the next combination
            else:
                print(f"No data found for '{keyword}' on {website}")

def main():
    # Read websites and keywords from files
    websites = read_list_from_file('websites.txt')  # List of websites to search
    keywords = read_list_from_file('categories.txt')  # List of keywords to search for

    # Fetch all results based on the websites and keywords
    fetch_all_results(websites, keywords)

if __name__ == "__main__":
    main()  # Run the main function


#### WEB Scrapping - version1.0

---

In [1]:
import pandas as pd
import os
from bs4 import BeautifulSoup
import requests
import re

def scrape_saved_files(results_dir):
    """Performs web scraping on the saved CSV files to extract product title and price."""
    
    scraped_data = []
    if not os.path.exists('./Scraped_Results'):
        os.makedirs('./Scraped_Results')

    for file_name in os.listdir(results_dir):
        if file_name.endswith('.csv'):
                
            file_path = os.path.join(results_dir, file_name)
            print(f"Scraping data from file: {file_path}")
            df = pd.read_csv(file_path)

            for _, row in df.iterrows():
                archive_url = row.get("linkToNoFrame")
                if archive_url:
                    try:
                        print("----------------------------")
                        print(f"Scraping data from: {archive_url}")
                        page = requests.get(archive_url)
                        soup = BeautifulSoup(page.content, 'html.parser')

                        product_title = "N/A"
                        price = "N/A"
                        
                        site = re.split(r'[_\.]', file_name)
                        if 'worten' in site[1]:
                            product_title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "N/A"
                            price = soup.find(class_='w-product__price__current').get_text(strip=True) if soup.find(class_='w-product__price__current') else "N/A"
                        
                        elif 'fnac' in site:
                            
                            # List of possible classes for title and price
                            title_classes = ['f-productHeader-Title', 'Article-desc']
                            price_classes = ['f-priceBox-price', 'userPrice']

                            # Find the product title
                            product_title = "N/A"
                            for title_class in title_classes:
                                title_element = soup.find(class_=title_class)
                                if title_element:
                                    product_title = title_element.get_text(strip=True)
                                    break

                            # Find the product price
                            price = "N/A"
                            for price_class in price_classes:
                                price_element = soup.find(class_=price_class)
                                if price_element:
                                    price = price_element.get_text(strip=True)
                                    break

                        # elif 'worten' in site:
                        #     product_title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "N/A"
                        #     price = soup.find(class_='w-product-price__main').get_text(strip=True) if soup.find(class_='w-product-price__main') else "N/A"
                        # elif 'radiopopular' in site:
                        #     product_title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "N/A"
                        #     price = soup.find(class_='current-price').get_text(strip=True) if soup.find(class_='current-price') else "N/A"
        
                        
                        # if 'mediamarkt' in site:
                        #     print('aqui2')
                        #     print(soup.find('h1').get_text())
                            
                        #     product_title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "N/A"
                        #     price = soup.find(class_='price clearfix').get_text(strip=True) if soup.find(class_='price clearfix') else "N/A"
                        
                        # elif 'kuantokusta' in site:
                        #     product_title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "N/A"
                        #     price = soup.find(class_='product-price').get_text(strip=True) if soup.find(class_='new-price only-price') else "N/A"



                        if product_title == "N/A" or "Substituição" in product_title or price == "N/A":
                            print(f"Skipping invalid product page: {archive_url}")
                            continue
                        
                        scraped_data.append({
                            "title": product_title,
                            "price": price,
                            "category": site[3],
                            "timestamp": row.get("tstamp"),
                            "website": site[1],
                            "linkToNoFrame": row.get("linkToNoFrame")
                        })

                        print("----------------------------\n\n")
                        
                    except Exception as e:
                        print(f"Error scraping {archive_url}: {e}")
                        continue

            # Save scraped data to a new CSV
            if scraped_data:
                scraped_df = pd.DataFrame(scraped_data)
                scraped_output_file = f"./Scraped_Results/scraped_{file_name}"
                scraped_df.to_csv(scraped_output_file, index=False)
                print(f"Scraped data saved to {scraped_output_file}")
                scraped_data.clear()


def main():
    scrape_saved_files('./Results2')

if __name__ == "__main__":
    main()  # Run the main function

Scraping data from file: ./Results2/www_fnac_pt_Smartphone.csv
----------------------------
Scraping data from: https://arquivo.pt/noFrame/replay/20200512085740/https://www.fnac.pt/SearchResult/ResultList.aspx?SCat=0%211&Search=Xiaomi&sft=1&sa=0
----------------------------


----------------------------
Scraping data from: https://arquivo.pt/noFrame/replay/20200903102841/https://www.fnac.pt/Xiaomi/m49259/w-4
----------------------------


----------------------------
Scraping data from: https://arquivo.pt/noFrame/replay/20200305083414/https://www.fnac.pt/Xiaomi/m49259/w-4
----------------------------


----------------------------
Scraping data from: https://arquivo.pt/noFrame/replay/20200903065239/https://www.fnac.pt/Samsung/m20066/w-4
----------------------------


----------------------------
Scraping data from: https://arquivo.pt/noFrame/replay/20200305060722/https://www.fnac.pt/Samsung/m20066/w-4
----------------------------


----------------------------
Scraping data from: http

# Bibliography

https://www.worten.pt 

https://www.fnac.pt

https://www.radiopopular.pt

https://mediamarkt.pt

https://www.kuantokusta.pt